# ML-10 — Content Action Playbook

Translating model scores into an operational queue with transparent reason codes and editorial workflows.

## 1. Ranked actions + reason codes

Generate prioritized action queue with reason codes.

In [1]:
import pandas as pd
import numpy as np
import os

data_path = "data/raw/content_refresh_anonymized.csv" if os.path.exists("data/raw/content_refresh_anonymized.csv") else "/content/FlyRank-Internship/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)
active = df[df["avg_position"] > 0].copy()

# Opportunity score
demand_s = np.log1p(active["search_volume"].fillna(0)) / np.log1p(active["search_volume"].max()) * 100
vis_s = np.log1p(active["impressions_90d"].fillna(0)) / np.log1p(active["impressions_90d"].max()) * 100
pos_s = np.where(active["avg_position"].between(4, 20), 100, np.where(active["avg_position"].between(1, 3), 40, 60))
stale_s = (active["days_since_last_update"].fillna(0) / active["days_since_last_update"].max()) * 100

active["refresh_score"] = (0.30 * demand_s + 0.25 * vis_s + 0.25 * pos_s + 0.20 * stale_s).round(1)

def assign_reason(row):
    if row["days_since_last_update"] > 300:
        return "STALE_CONTENT"
    elif row["avg_position"] > 4 and row["avg_position"] <= 20:
        return "RANKING_DROP"
    else:
        return "HIGH_DEMAND_DECLINE"

active["reason_code"] = active.apply(assign_reason, axis=1)
active["action"] = active["reason_code"].map({
    "STALE_CONTENT": "Comprehensive Content Overhaul",
    "RANKING_DROP": "On-Page Snippet & Heading Optimization",
    "HIGH_DEMAND_DECLINE": "Urgent Refresh & Diagnostic"
})

queue = active.sort_values("refresh_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

display(queue[["rank", "content_id", "refresh_score", "reason_code", "action", "avg_position", "impressions_90d"]].head(10))


,rank,content_id,refresh_score,reason_code,action,avg_position,impressions_90d
0,1,content_5fe46e04994d,75.8,RANKING_DROP,On-Page Snippet & Heading Optimization,4.2,517715
1,2,content_2db251d1a841,74.9,RANKING_DROP,On-Page Snippet & Heading Optimization,5.6,198671
2,3,content_2e0b3dc70916,74.6,RANKING_DROP,On-Page Snippet & Heading Optimization,6.9,27948
3,4,content_aaef01a50def,73.6,RANKING_DROP,On-Page Snippet & Heading Optimization,5.4,517109
4,5,content_2725d2bcfac1,73.5,RANKING_DROP,On-Page Snippet & Heading Optimization,9.1,27348
5,6,content_eb1510f4b5f1,72.9,RANKING_DROP,On-Page Snippet & Heading Optimization,14.7,12275
6,7,content_e6955a2c59dc,72.5,RANKING_DROP,On-Page Snippet & Heading Optimization,7.1,37534
7,8,content_8ca50876b0df,72.0,RANKING_DROP,On-Page Snippet & Heading Optimization,18.3,8699
8,9,content_2c2606c5d176,71.9,RANKING_DROP,On-Page Snippet & Heading Optimization,4.2,347399
9,10,content_496544bf85aa,71.9,RANKING_DROP,On-Page Snippet & Heading Optimization,5.0,22668


## 2. Intended use and limits

Operational guidelines for editorial and SEO teams.

In [2]:
print("INTENDED USE:")
print("1. Weekly editorial backlog triage.")
print("2. SEO snippet optimization prioritization.")
print("LIMITS: Does not automate content generation or guarantee ranking recovery.")


INTENDED USE:
1. Weekly editorial backlog triage.
2. SEO snippet optimization prioritization.
LIMITS: Does not automate content generation or guarantee ranking recovery.


## 3. Human review + the no-go list

Exceptions and guardrails before editing content.

In [3]:
no_go_rules = pd.DataFrame([
    {"Rule": "Recently Edited Pages", "Condition": "Edited in last 30 days", "Action": "Skip / Cool-off period"},
    {"Rule": "Core Brand Pages", "Condition": "Brand navigation queries", "Action": "Exclude from algorithmic changes"},
    {"Rule": "Seasonal Hubs", "Condition": "Off-season temporary drop", "Action": "Monitor until in-season"}
])
display(no_go_rules)


,Rule,Condition,Action
0,Recently Edited Pages,Edited in last 30 days,Skip / Cool-off period
1,Core Brand Pages,Brand navigation queries,Exclude from algorithmic changes
2,Seasonal Hubs,Off-season temporary drop,Monitor until in-season


## 4. Monitoring / retrain triggers

Conditions that require model recalibration.

In [4]:
print("MONITORING TRIGGERS:")
print("- Major search engine core algorithm updates")
print("- Substantial seasonal traffic shift across client base")
print("- CMS migration or URL structure changes")


MONITORING TRIGGERS:
- Major search engine core algorithm updates
- Substantial seasonal traffic shift across client base
- CMS migration or URL structure changes


## 5. Exports for the paper

Save the final action queue and metric summary.

In [5]:
os.makedirs("work/outputs", exist_ok=True)
export_cols = ["rank", "content_id", "refresh_score", "reason_code", "action", "avg_position", "impressions_90d"]
queue[export_cols].head(100).to_csv("work/outputs/action_playbook_queue.csv", index=False)
print("Wrote work/outputs/action_playbook_queue.csv (Top 100 sample)")


Wrote work/outputs/action_playbook_queue.csv (Top 100 sample)


## Self-check

- [x] Ranked queue generated with reason codes
- [x] Human review no-go rules defined
- [x] Outputs exported for research paper